# Advocate Health CRIO Phenotype Library
## Investigator Usage Guide

This notebook walks through the complete lifecycle of a computable phenotype —
from project initialization through validation, library deposit, and export
to the OHDSI Phenotype Library.

Every phenotype in this library was produced using the `crio` Python library,
which enforces the metadata schema, IRB requirements, and contribution economy
that govern access to Advocate Health secure computing environments.

Run each cell in sequence. The notebook uses a temporary directory so nothing
is written to the library until the explicit commit step.

## 0. Setup

Install `crio` if not already installed:

```bash
pip install git+https://github.com/advocate-phenotype-dev/crio-py
```

In [ ]:
import tempfile
from pathlib import Path
import crio

print(f"crio version: {crio.__version__}")

# working directory for this demo
tmp = Path(tempfile.mkdtemp())
print(f"Working directory: {tmp}")

## 1. Initialize a new phenotype project

`crio.init()` creates a UUID-named project directory with a stub structure
and writes `advocate-phenotype.yaml` — the machine-validated source of truth
for the project.

Every field is validated against the schema at commit time. The IRB number
and status are required for SCE tier 3 and above — the system enforces this,
not the administrator.

In [ ]:
project_dir = crio.init(
    pi_name="Jane Reyes",
    pi_orcid="0000-0000-0000-0001",
    pi_email="jane.reyes@advocatehealth.org",
    department="Cardiovascular Epidemiology",
    phenotype_name="Atrial Fibrillation — Incident Cases",
    domain="condition",
    sce_tier=3,
    data_tier="B",
    environment="azure_tre",
    omop_aligned=True,
    clarity_required=False,
    description=(
        "Identifies adult patients with a new diagnosis of atrial fibrillation (AF) "
        "with no prior AF diagnosis in the preceding 24 months. "
        "Intended for incident AF cohort assembly for epidemiological research "
        "and trial-ready cohort identification across the Advocate Health network."
    ),
    inclusion_criteria=(
        "Age >= 18 at index date; "
        "ICD-10-CM I48.x in condition_occurrence; "
        "No I48.x diagnosis in 24 months prior to index date; "
        "At least one encounter in the 12 months prior to index date; "
        "OMOP concept ID 313217 (atrial fibrillation)."
    ),
    exclusion_criteria=(
        "Age < 18; "
        "Prior AF diagnosis within 24 months of index date; "
        "AF diagnosed only in the context of cardiac surgery (procedure_occurrence "
        "with OMOP concept 4195448 within 30 days of index date); "
        "Enrollment in hospice at index date."
    ),
    irb_number="IRB-2026-042",
    irb_status="active",
    funding_source="R01-HL-XXXXXX",
    output_dir=tmp,
)

print(f"Project directory: {project_dir}")
print(f"Project ID: {project_dir.name}")

## 2. Inspect the generated schema

The `advocate-phenotype.yaml` file is the canonical record for this project.
The project UUID is system-assigned and immutable. The `updated` timestamp
is refreshed on every commit. `deposit_eligible` is computed — the investigator
cannot self-certify.

In [ ]:
import yaml

with open(project_dir / "advocate-phenotype.yaml") as f:
    schema = yaml.safe_load(f)

print(yaml.dump(schema, default_flow_style=False, sort_keys=False))

## 3. Inspect the stub directory structure

The project directory is pre-structured for the work that needs to happen:
cohort definitions go in `src/cohort_definition/`, Clarity-direct SQL in
`src/clarity_queries/`, validation outputs in `src/validation/`.
Nothing in `.advocate/` ever commits — it holds session credentials only.

In [ ]:
import os

for root, dirs, files in os.walk(project_dir):
    dirs[:] = [d for d in dirs if d not in (".git", ".advocate")]
    level = root.replace(str(project_dir), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    for f in files:
        print(f"{indent}  {f}")

## 4. Start a working session

`crio.source()` validates the schema, writes session credentials to
`.advocate/session.lock` (gitignored), and sets environment variables
for the session. In sandbox mode it returns mock credentials.
In production it will handshake with the Azure TRE credential endpoint.

In [ ]:
session = crio.source(project_dir, sandbox=True)
print(f"\nEnvironment variables set:")
import os
for key in ["CRIO_PROJECT_ID", "CRIO_SCE_TIER", "CRIO_ENVIRONMENT", "CRIO_SANDBOX"]:
    print(f"  {key} = {os.environ.get(key)}")

## 5. Validate the schema

Validation runs the full Pydantic model. It checks field formats,
conditional requirements, and computes deposit eligibility.
A new project is `draft` status — not yet deposit eligible.

In [ ]:
report = crio.validate(project_dir)
report.print_report()

## 6. Update validation status

After running CohortDiagnostics against the NEXUS/OMOP layer and confirming
the cohort definition performs as expected, the investigator updates the
validation status and adds OMOP concept IDs. Deposit eligibility is
recomputed automatically on the next validation call.

In [ ]:
schema_path = project_dir / "advocate-phenotype.yaml"

with open(schema_path) as f:
    raw = yaml.safe_load(f)

raw["phenotype"]["validation_status"] = "internal_validated"
raw["phenotype"]["validation_method"] = "cohort_diagnostics"
raw["phenotype"]["target_concept_ids"] = [
    313217,   # Atrial fibrillation (SNOMED)
    4068155,  # Atrial flutter (SNOMED)
]
raw["phenotype"]["icd_codes"] = [
    "I48.0",  # Paroxysmal atrial fibrillation
    "I48.11", # Longstanding persistent AF
    "I48.19", # Other persistent AF
    "I48.20", # Chronic AF unspecified
    "I48.21", # Permanent AF
]

with open(schema_path, "w") as f:
    yaml.dump(raw, f, default_flow_style=False, sort_keys=False)

print("Schema updated")

## 7. Re-validate — confirm deposit eligibility

With `internal_validated` status and all required fields populated,
the validator now sets `deposit_eligible: true` automatically.

In [ ]:
report = crio.validate(project_dir)
report.print_report()

## 8. Commit to the phenotype library

`crio.commit()` validates the schema, updates the timestamp, regenerates
the README, copies the project into the library under its UUID, updates
`registry.yaml`, and commits to the phenotype-library Git repository.

The commit message is prefixed with `[crio]` and includes the phenotype
name, version, and truncated UUID for traceability.

In [ ]:
import git

# point at the actual phenotype-library repo
library_dir = Path("__file__").parent.parent

# for demo purposes use a temp library
demo_library = tmp / "phenotype-library"
demo_library.mkdir()
git.Repo.init(demo_library)
(demo_library / "projects").mkdir()

import yaml as _yaml
with open(demo_library / "registry.yaml", "w") as f:
    _yaml.dump({"generated": "", "projects": []}, f)

crio.commit(
    project_dir=project_dir,
    library_dir=demo_library,
    message="Incident AF phenotype v0.1.0 — initial deposit",
    sandbox=True,
)

## 9. Inspect the registry

`registry.yaml` is the machine-generated index the clearinghouse reads
to determine queue position, contribution credit, and deposit eligibility
across all projects. It is never hand-edited.

In [ ]:
with open(demo_library / "registry.yaml") as f:
    registry = yaml.safe_load(f)

print(yaml.dump(registry, default_flow_style=False, sort_keys=False))

## 10. Export to OHDSI Phenotype Library format

`crio.export()` transforms the `advocate-phenotype.yaml` into the
OHDSI PL JSON format for external submission. OMOP-aligned phenotypes
from Advocate are deposited to the OHDSI Phenotype Library with
institutional attribution and a DOI assigned per version.

In [ ]:
ohdsi_export = crio.export(project_dir, target="ohdsi_pl")

## 11. What happens next

Once committed and deposit-eligible:

1. **Clearinghouse review** — the CRIO office reviews the deposit against
   the instrument definition. A qualifying deposit earns contribution credit.

2. **Contribution credit** — credit is logged against the PI's ORCID iD
   and applied as queue priority for future SCE access requests.
   A deposit at SCE-3 exit advances the PI three positions in the
   subsidized access queue.

3. **OHDSI PL submission** — OMOP-aligned phenotypes are submitted
   to the OHDSI Phenotype Library. The PI receives a DOI and is credited
   via ORCID. Downstream investigators cite the phenotype directly.

4. **Reuse** — future investigators building on this phenotype reference
   it in their `advocate-phenotype.yaml` via `phekb_id` or `ohdsi_pl_id`.
   The provenance chain is maintained in Git history.

---

**Questions or access requests:** contact the CRIO office or open an issue
at https://github.com/advocate-phenotype-dev/crio-py